In [10]:
# 5.1.3 Model 3: XGBoost Regressor
# Trained on X_train.csv (unscaled) since tree-based models don't need feature scaling.

import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import json
from sklearn.model_selection import GridSearchCV
import sys, os

sys.path.append(os.path.dirname(os.path.abspath('__file__')))
from model_utils import evaluate_model, cross_validate_model

MODELLING_DIR = os.path.join("..", "data", "modelling")
MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

X_train = pd.read_csv(os.path.join(MODELLING_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(MODELLING_DIR, "X_test.csv"))

# price is already log-transformed in the CSV (see Section 3.6) — read as-is
y_train = pd.read_csv(os.path.join(MODELLING_DIR, "y_train.csv"))["price"]
y_test = pd.read_csv(os.path.join(MODELLING_DIR, "y_test.csv"))["price"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (3021, 52) | X_test: (756, 52)


In [11]:
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
}

grid_search = GridSearchCV(
    xgb.XGBRegressor(random_state=42, objective="reg:squarederror"),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print(f"Best CV score (log-scale RMSE): {-grid_search.best_score_:.4f}")

# save tuning result for reference/reproducibility
with open(os.path.join(MODEL_DIR, "xgboost_best_params.json"), "w") as f:
    json.dump(grid_search.best_params_, f, indent=2)

Best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 400, 'subsample': 0.8}
Best CV score (log-scale RMSE): 0.2487


In [12]:
model = xgb.XGBRegressor(**grid_search.best_params_, random_state=42, objective="reg:squarederror")
model.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [13]:
train_metrics = evaluate_model(model, X_train, y_train, label="Train")
print()
test_metrics = evaluate_model(model, X_test, y_test, label="Test")

# save metrics for report reference
with open(os.path.join(MODEL_DIR, "xgboost_metrics.json"), "w") as f:
    json.dump({"train": train_metrics, "test": test_metrics}, f, indent=2)

Train RMSE:  RM 48,785  (13.9% of median price)
Train MAE:   RM 30,663
Train MAPE:  8.0%
Train R2:    0.9794
Train MSE:   2,379,985,674

Test RMSE:  RM 174,671  (49.2% of median price)
Test MAE:   RM 76,164
Test MAPE:  17.4%
Test R2:    0.6254
Test MSE:   30,510,015,121


In [14]:
cv_results = cross_validate_model(
    xgb.XGBRegressor(**grid_search.best_params_, random_state=42, objective="reg:squarederror"),
    X_train, y_train, n_splits=5,
)

5-fold CV (mean +/- std):
  RMSE:  RM 178,411 +/- 20,792  (50.8% of median price)
  MAE:   RM 77,025 +/- 2,449
  MAPE:  18.0% +/- 1.1%
  R2:    0.7180 +/- 0.0400
  MSE:   32,262,676,774 +/- 6,943,332,206


In [15]:
importance_table = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 15 features by importance:")
print(importance_table.head(15))

Top 15 features by importance:
Has_Gymnasium                     0.155147
Has_Security                      0.074012
State_Penang                      0.073587
PropertyType_Flat                 0.065724
Property Size                     0.064887
PropertyType_Service_Residence    0.039878
Property_Age_Missing              0.035836
State_Perak                       0.030010
State_Selangor                    0.029746
State_Melaka                      0.029552
PropertyType_Condominium          0.026986
Parking Lot                       0.026304
Bathroom                          0.025773
Total_Units_Missing               0.024252
State_Sabah                       0.020489
dtype: float32


In [16]:
model_path = os.path.join(MODEL_DIR, "xgboost_model.pkl")
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

Model saved to ..\models\xgboost_model.pkl


In [17]:
# Demo: reload the saved model and predict on an existing test row
import joblib
model = joblib.load(os.path.join(MODEL_DIR, "xgboost_model.pkl"))

sample = X_test.iloc[[0]]           # first row of X_test as a stand-in example
preds_log = model.predict(sample)
preds_rm = np.exp(preds_log)

print(f"Predicted price: RM {preds_rm[0]:,.0f}")
print(f"Actual price:    RM {np.exp(y_test.iloc[0]):,.0f}")

Predicted price: RM 318,285
Actual price:    RM 290,000


In [18]:
print("y_test head (log-price):", y_test.head())
print("y_test dtype:", y_test.dtype)
print("Expected actual price (RM):", np.exp(y_test.iloc[0]))

y_test head (log-price): 0    12.577636
1    13.458836
2    12.570716
3    13.527828
4    12.144197
Name: price, dtype: float64
y_test dtype: float64
Expected actual price (RM): 289999.9999999999
